# 📦 Batch Inference (many images → grid + CSV) — One-click Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/AnyDisasterMapping/blob/main/batch_inference_colab.ipynb)
[![GitHub](https://img.shields.io/badge/Repo-AnyDisasterMapping-181717?logo=github)](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping)
![Task](https://img.shields.io/badge/task-seg%20%2B%20cd-4c8cbf) ![Auto](https://img.shields.io/badge/runs-automatically-3fb950)

Throw a whole folder of scenes at the pipeline: multi-select uploads, automatic **pre/post pairing** by filename (`_pre` / `_post`), per-image masks + statistics, a contact-sheet grid, and a downloadable `batch_summary.csv`. Singles are segmented, pairs get change detection — all clearly mode-labeled.

| ▶️ How to run | |
|---|---|
| **Everything automatic** | Click **Runtime ▸ Run all** (Ctrl+F9): setup → demo scene → inference → dashboard → export chain without any further clicks |
| **With your data** | Run the cells once, then click the 📤 upload buttons — the pipeline re-runs automatically after upload |
| **No image at hand?** | The 🛟 fallback generates a clearly-labeled synthetic flood scene — the notebook *never blocks or crashes* |

**📤 Upload preview — what to have ready:**

| file | what it is | typical size |
|---|---|---|
| **`any_images...png`** | multi-select: singles OR *_pre/*_post pairs | 512–2048 px each |
| **`pairing rule`** | same name with _pre/_post suffix → change detection | — |
| **`no upload?`** | 6 synthetic scenes across domains are generated | — |

> 🔖 Every result is stamped with its mode: **CHECKPOINT** (your `.pth`), **QUICK-TRAIN** (genuinely briefly-trained), or **SYNTHETIC-DEMO** (clearly-labeled heuristic). Nothing is faked.

In [ ]:
#@title ⚙️ SETUP — auto-installs, idempotent, restart-safe {display-mode:"form"}
AUTO_RUN = True  #@param {type:"boolean"}
SHOW_PREVIEW_ON_UPLOAD = True  #@param {type:"boolean"}
import os, sys, glob, json, math, random, subprocess, traceback, datetime, textwrap

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

colab_kit.pip_install("ipywidgets", "pillow", "numpy", "imageio", "scikit-image", "tifffile", "matplotlib")

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageDraw
import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output
from colab_kit import (make_synthetic, infer, overlay, heatmap, to_u8, stats_for,
                       stylize_poster, read_any, verdict, show_pair, run_inference,
                       run_dashboard, export_results_zip_a, nb_save_upload, nb_apply,
                       nb_upload_pre, nb_upload_post, nb_use_synthetic,
                       _field, _fbm, _stretch, _blur, _median_mask, _terrain, _hsv,
                       segment_heuristic, change_heuristic, _otsu)

random.seed(42); np.random.seed(42)
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
DOMAIN = "flood"          # this notebook's disaster domain
TASK   = "both"            # "seg" | "cd" | "both"
GSD_M  = 10.0               # ground sample distance (m/pixel) for km² stats
NOTEBOOK = "batch_inference_colab.ipynb"
STATE = {"domain": DOMAIN, "task": "seg" if TASK != "cd" else "cd",
         "pre": None, "post": None, "gt": None, "mask": None, "source": "none"}
print("✅ Environment ready — notebook:", NOTEBOOK, "| domain:", DOMAIN, "| task:", TASK)

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


In [ ]:
#@title ☁️ MOUNT GOOGLE DRIVE — persistent uploads, checkpoints & results {display-mode:"form"}
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT = "AnyDisasterMapping"  #@param {type:"string"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

DRIVE_ROOT, IN_COLAB = colab_kit.mount_and_prepare(DRIVE_PROJECT, MOUNT_DRIVE)

In [ ]:
#@title 📤 UPLOAD — buttons + inline preview, never blocks {display-mode:"form"}
# upload logic lives in colab_kit (single source of truth, unit-tested in CI)
def _save_upload(name, data):
    return colab_kit.nb_save_upload(globals(), name, data)

def _apply(kind, arr):
    return colab_kit.nb_apply(globals(), kind, arr)

def upload_pre(_=None):
    return colab_kit.nb_upload_pre(globals())

def upload_post(_=None):
    return colab_kit.nb_upload_post(globals())

def use_synthetic(_=None):
    return colab_kit.nb_use_synthetic(globals())


SPEC = [['any_images...png',
  'multi-select: singles OR *_pre/*_post pairs',
  '512–2048 px each'],
 ['pairing rule', 'same name with _pre/_post suffix → change detection', '—'],
 ['no upload?', '6 synthetic scenes across domains are generated', '—']]
W = 70
def print_spec():
    print("┌" + "─" * W + "┐")
    for line in textwrap.wrap("📤 WHAT TO UPLOAD — " + NOTEBOOK, W - 2):
        print("│ " + line.ljust(W - 2) + "│")
    for label, desc, size in SPEC:
        wrapped = textwrap.wrap(label + " — " + desc + "  (" + size + ")", W - 6)
        for j, line in enumerate(wrapped):
            print(("│   • " if j == 0 else "│     ") + line.ljust(W - 6) + "│")
    for line in textwrap.wrap(
            "accepted: .png .jpg .jpeg .tif .tiff .h5 | sweet spot: 512–2048 px, ≤ 20 MB each | "
            "GeoTIFF georeference is kept when present", W - 2):
        print("│ " + line.ljust(W - 2) + "│")
    print("└" + "─" * W + "┘")
print_spec()

out_up = widgets.Output(layout={"border": "1px solid #ddd"})
domain_sel = widgets.Dropdown(
    options=["flood", "landslide", "wildfire", "oil_spill", "avalanche",
             "volcano", "glof", "dust_storm", "smoke_plume", "locust",
             "tsunami", "hurricane", "infra_damage"],
    value="flood", description="Domain:")
def _sync_dom(*_):
    STATE["domain"] = domain_sel.value
domain_sel.observe(_sync_dom, names="value")
task_sel = widgets.ToggleButtons(
    options=[("🖼️ single image (segmentation)", "seg"),
             ("🆚 pre/post pair (change detection)", "cd")],
    value="seg", description="Task:")
def _sync_task(*_):
    STATE["task"] = task_sel.value
    btn_post.layout.display = None if task_sel.value == "cd" else "none"
task_sel.observe(_sync_task, names="value")
gsd_box = widgets.FloatText(value=GSD_M, description="GSD m/px:",
                            tooltip="ground sample distance, used for km² statistics")
btn_post = widgets.Button(description="⬆️ Upload POST image", button_style="success", icon="upload")
btn_extra = None

btn_extra = widgets.Button(description="⬆️ Upload MANY images (multi-select)",
                           button_style="info", icon="upload")
def upload_many(_=None):
    with out_up:
        clear_output()
        print("📂 choose one or many images (Ctrl/Cmd-click) …")
    try:
        for name, data in _gfiles.upload().items():
            p = os.path.join(OUT_DIR, "uploads", name)
            os.makedirs(os.path.dirname(p), exist_ok=True)
            with open(p, "wb") as f:
                f.write(data)
            arr = read_any(p)
            STATE.setdefault("batch", []).append((name, arr))
            with out_up:
                verdict(p, arr)
        with out_up:
            print("✅", len(STATE.get("batch", [])), "file(s) queued — RUN processes them all")
            if AUTO_RUN and "run_batch" in globals():
                run_batch()
    except Exception as e:
        with out_up:
            print("upload failed:", e)
btn_extra.on_click(upload_many)


btn_pre = widgets.Button(description="⬆️ Upload PRE image", button_style="primary", icon="upload")
btn_synth = widgets.Button(description="🛟 Use bundled synthetic sample",
                           button_style="warning", icon="picture")
btns = [btn_pre]
if TASK in ("cd", "both"):
    btns.insert(1, btn_post)
if btn_extra is not None:
    btns.append(btn_extra)
box = widgets.VBox([widgets.HBox(btns), widgets.HBox([btn_synth, gsd_box]), out_up])






btn_pre.on_click(upload_pre)
if TASK in ("cd", "both"):
    btn_post.on_click(upload_post)
btn_synth.on_click(use_synthetic)
display(box)

if AUTO_RUN and STATE["pre"] is None:
    print("⚡ AUTO_RUN: generating bundled synthetic sample — upload later to re-run automatically")
    use_synthetic()

In [ ]:
#@title ▶️ RUN BATCH — pairing, grid, CSV summary {display-mode:"form"}
MAX_PANELS = 12  #@param {type:"integer"}
MODE_TAG = "BATCH"

def _pair(raw):
    pres, posts, singles = {}, {}, []
    for name, arr in raw:
        stem = os.path.splitext(name)[0].lower()
        if stem.endswith("_post") or stem.endswith("-post"):
            posts[stem.replace("_post", "").replace("-post", "")] = arr
        elif stem.endswith("_pre") or stem.endswith("-pre"):
            pres[stem.replace("_pre", "").replace("-pre", "")] = arr
        else:
            singles.append((name, arr))
    pairs = []
    for k, v in pres.items():
        if k in posts:
            pairs.append((k, v, posts[k]))
        else:
            singles.append((k + "_pre", v))
    return pairs, singles

def run_batch():
    STATE["mode_tag"] = MODE_TAG
    import csv
    raw = list(STATE.get("batch") or [])
    if not raw:
        print("🛟 no uploads queued — synthesizing 6 demo pairs across flagship domains")
        for d in ["flood", "wildfire", "tsunami", "hurricane", "landslide", "volcano"]:
            pre, post, _gt = make_synthetic(d, 384, 384, seed=random.randint(0, 9999))
            raw.append((d + "_demo_pre.png", pre))
            raw.append((d + "_demo_post.png", post))
    pairs, singles = _pair(raw)
    jobs = [(n, "cd", a, b) for n, a, b in pairs] + [(n, "seg", None, a) for n, a in singles]
    outs = os.path.join(OUT_DIR, "batch")
    os.makedirs(outs, exist_ok=True)
    rows, grid = [], []
    for name, task, pre, post in jobs:
        if task == "cd":
            mask, tag = infer(STATE["domain"], "cd", pre, post)
        else:
            mask, tag = infer(STATE["domain"], "seg", post)
        st = stats_for(mask, STATE.get("gsd", GSD_M))
        rows.append({"file": name, "task": task, "mode": tag, **st})
        ov = overlay(post, mask)
        safe = "ov_" + name if os.path.splitext(name)[1] else "ov_" + name + ".png"
        Image.fromarray(to_u8(ov)).save(os.path.join(outs, safe))
        if len(grid) < MAX_PANELS:
            grid.append((name, task, tag, ov))
    with open(os.path.join(OUT_DIR, "batch_summary.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file", "task", "mode", "changed_px",
                                          "changed_pct", "km2"])
        w.writeheader()
        for r in rows:
            w.writerow(r)
    n = len(grid)
    if n:
        cols = min(4, n)
        rws = math.ceil(n / cols)
        fig, ax = plt.subplots(rws, cols, figsize=(3.8 * cols, 3.4 * rws))
        flat = np.atleast_1d(ax).ravel()
        for a, (name, task, tag, ov) in zip(flat, grid):
            a.imshow(np.clip(ov, 0, 1))
            a.set_title(name + "\n" + task + " · " + tag, fontsize=7)
            a.axis("off")
        for a in flat[n:]:
            a.axis("off")
        plt.suptitle("batch results (first " + str(n) + " of " + str(len(rows)) + ")")
        plt.show()
    print("📋 summary →", os.path.join(OUT_DIR, "batch_summary.csv"))
    for r in rows:
        print("   ", r["file"], "|", r["task"], "|", r["mode"], "|", r["changed_pct"], "%")
    if AUTO_RUN and "run_export" in globals():
        run_export()

run_batch() if AUTO_RUN else print("upload images or call run_batch()")

In [ ]:
#@title 📦 EXPORT — results zip + optional Drive copy + auto-download {{display-mode:"form"}}
USE_DRIVE = False  #@param {{type:"boolean"}}

def run_export():
    return colab_kit.export_results_zip_a(globals(), use_drive=USE_DRIVE)

run_export() if AUTO_RUN else print("call run_export() when ready")

## 🧾 Output contract

| artifact | meaning |
|---|---|
| `outputs/batch/ov_<file>.png` | per-image overlay (red = detected class / change) |
| `outputs/batch_summary.csv` | file, task, mode tag, changed_px, changed %, km² |
| contact sheet | first 12 results, one panel each |
| export zip | everything above + manifest with mode tags |

> Pairs are detected by `_pre` / `_post` suffixes (also accepts `-pre` / `-post`). Everything without a pair is segmented as a single image.